# 病原菌 Raman 训练

按“环境 → 数据 → 实验组 → 训练 → 评估 → 分析 → 打包”的顺序执行；每个代码格只保留配置和短调用。

## 1. 环境与数据包

In [ ]:
"""运行环境引导：定位项目根目录、校验依赖、报告设备信息。"""

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# None 表示自动探测
PROJECT_ROOT_OVERRIDE = None
# 缺失依赖时自动 pip 安装；网络受限时设为 False，按提示手动安装
INSTALL_MISSING_DEPS = True
# AutoDL 默认清华镜像；Colab/海外可设为 None 使用默认 PyPI
PIP_INDEX_URL = (
    os.environ.get("PIP_INDEX_URL")
    or os.environ.get("RAMAN_PIP_INDEX_URL")
    or "https://pypi.tuna.tsinghua.edu.cn/simple"
)
# umap-learn 会拉取 numba/llvmlite，首次安装可能较慢；仅嵌入分析需要。
INSTALL_UMAP_DEPS = True
PIP_TIMEOUT_SECONDS = 900

PACKAGE_ARCHIVE_NAME = "ramanv2.zip"
DATA_ARCHIVE_NAME = "data.npz"
# 训练、评估、分析链路需要的第三方模块 -> pip 包名。
REQUIRED_MODULES = {
    "torch": "torch",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "yaml": "PyYAML",
    "openpyxl": "openpyxl",
}
# 嵌入分析才需要的可选依赖，避免首次运行被 numba/llvmlite 编译阻塞。
OPTIONAL_MODULES = {"umap": "umap-learn"}


def is_module_available(name: str) -> bool:
    """安全探测模块能否导入；find_spec 对多级名称在父包缺失时会抛异常。"""
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def has_project_payload(directory: Path) -> bool:
    """目录是否已含可用代码包：ramanv2.zip，或含 core/config.py 的 ramanv2/ 包。"""
    if (directory / PACKAGE_ARCHIVE_NAME).is_file():
        return True
    return (directory / "ramanv2" / "core" / "config.py").is_file()


def resolve_project_root() -> Path:
    """按 环境变量/显式覆盖 -> 工作目录及其祖先 的顺序定位项目根目录。"""
    override = PROJECT_ROOT_OVERRIDE or os.environ.get("RAMAN_PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        root.mkdir(parents=True, exist_ok=True)
        return root
    working_dir = Path.cwd().resolve()
    for candidate in (working_dir, *working_dir.parents):
        if has_project_payload(candidate):
            return candidate
    return working_dir


def _pip_command(packages: list[str]) -> list[str]:
    """构建带镜像、超时和进度显示的 pip 命令。"""
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--progress-bar",
        "on",
        "--timeout",
        "60",
        "--retries",
        "2",
    ]
    if PIP_INDEX_URL:
        command.extend(["--index-url", PIP_INDEX_URL])
    return command + list(packages)


def _install_packages(packages: list[str]) -> bool:
    """执行 pip 安装；失败或超时时打印可复制的手动命令。"""
    command = _pip_command(packages)
    print("执行：" + " ".join(command))
    try:
        subprocess.run(
            command,
            shell=False,
            check=True,
            text=True,
            encoding="utf-8",
            timeout=PIP_TIMEOUT_SECONDS,
        )
        return True
    except subprocess.TimeoutExpired:
        print(f"pip 超过 {PIP_TIMEOUT_SECONDS} 秒未完成，已停止等待。")
    except subprocess.CalledProcessError as exc:
        print(f"pip 安装失败（exit={exc.returncode}）。")
    print("请在终端手动执行以下命令，然后重启 kernel：")
    print("  " + " ".join(command))
    return False


def ensure_dependencies(install_enable: bool) -> list[str]:
    """检查必需模块并按需 pip 安装，返回仍然缺失的核心模块名。"""
    missing = [name for name in REQUIRED_MODULES if not is_module_available(name)]
    optional_missing = [name for name in OPTIONAL_MODULES if not is_module_available(name)]
    if missing and install_enable:
        packages = [REQUIRED_MODULES[name] for name in missing]
        print(f"安装缺失依赖：{', '.join(packages)}")
        _install_packages(packages)
        missing = [name for name in REQUIRED_MODULES if not is_module_available(name)]
    elif missing:
        print(f"缺少依赖（未自动安装）：{', '.join(REQUIRED_MODULES[name] for name in missing)}")
        print("可手动执行：")
        print("  " + " ".join(_pip_command([REQUIRED_MODULES[name] for name in missing])))
    if optional_missing and install_enable and INSTALL_UMAP_DEPS:
        packages = [OPTIONAL_MODULES[name] for name in optional_missing]
        print(f"安装可选依赖：{', '.join(packages)}")
        _install_packages(packages)
        optional_missing = [name for name in OPTIONAL_MODULES if not is_module_available(name)]
    if optional_missing:
        print(
            "可选依赖未安装（仅嵌入分析需要）："
            + ", ".join(OPTIONAL_MODULES[name] for name in optional_missing)
        )
    return missing


PROJECT_ROOT = resolve_project_root()
still_missing_modules = ensure_dependencies(INSTALL_MISSING_DEPS)
print(f"project_root = {PROJECT_ROOT}")
print(f"pip_index    = {PIP_INDEX_URL or 'default PyPI'}")
if still_missing_modules:
    print(f"缺少依赖（需手动安装）：{', '.join(still_missing_modules)}")

import shutil as shutil_module

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "缺少 torch：AutoDL 镜像通常自带 CUDA 版 torch；请按实例 CUDA 版本手动安装后重启 kernel。"
    ) from exc

print(f"python       = {sys.version.split()[0]} ({sys.executable})")
print(f"torch        = {torch.__version__}, cuda_build = {torch.version.cuda}")
print(f"cpu_count    = {os.cpu_count()}")
print(
    f"disk_free    = {shutil_module.disk_usage(str(PROJECT_ROOT)).free / 2**30:.1f} GiB"
    f" @ {PROJECT_ROOT}"
)
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(
            f"gpu[{index}]      = {properties.name}, "
            f"cc={properties.major}.{properties.minor}, "
            f"vram={properties.total_memory / 2**30:.1f} GiB"
        )
    print(f"bf16_support = {torch.cuda.is_bf16_supported()}")
else:
    print("CUDA 不可用：训练与评估会回退到 CPU，use_amp 自动关闭。")
    print("AutoDL: 请确认选择的是 GPU 实例，并检查 torch 是否为 CUDA 版本。")

In [ ]:
# 输入检查：代码包必需；data.npz 可选，dataset/<数据集>/ 已在位时可跳过恢复。
REQUIRED_ARCHIVES = {"ramanv2.zip", DATA_ARCHIVE_NAME}
absent_archives = sorted(
    name for name in REQUIRED_ARCHIVES if not (PROJECT_ROOT / name).is_file()
)
if PACKAGE_ARCHIVE_NAME in absent_archives and not (PROJECT_ROOT / "ramanv2").is_dir():
    raise FileNotFoundError(
        f"缺少代码包：请把 {PACKAGE_ARCHIVE_NAME} 或解压后的 ramanv2/ 放到 {PROJECT_ROOT}"
    )
if DATA_ARCHIVE_NAME in absent_archives:
    print(f"未找到 {DATA_ARCHIVE_NAME}：若 dataset/<数据集>/init 已在位可忽略。")
print("输入检查完成。")


In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

PACKAGE_ARCHIVE = PROJECT_ROOT / PACKAGE_ARCHIVE_NAME
PACKAGE_DIR = PROJECT_ROOT / "ramanv2"


def load_package():
    """检测到压缩包时替换 ramanv2；否则使用已解压的包。"""
    if PACKAGE_ARCHIVE.is_file():
        temp_dir = Path(tempfile.mkdtemp(prefix="ramanv2_unpack_", dir=PROJECT_ROOT))
        try:
            with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
                archive.extractall(temp_dir)
            unpacked_dir = temp_dir / PACKAGE_DIR.name
            if not unpacked_dir.is_dir():
                raise FileNotFoundError("压缩文件中未找到 ramanv2/ 目录")
            if PACKAGE_DIR.exists():
                if not PACKAGE_DIR.is_dir():
                    raise FileExistsError(f"包路径不是目录：{PACKAGE_DIR}")
                shutil.rmtree(PACKAGE_DIR)
            shutil.move(str(unpacked_dir), str(PROJECT_ROOT))
            print(f"replaced package: {PACKAGE_ARCHIVE.name} -> {PACKAGE_DIR}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
    elif not PACKAGE_DIR.is_dir():
        raise FileNotFoundError(f"请先将 {PACKAGE_ARCHIVE_NAME} 放到 {PROJECT_ROOT}")
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))


load_package()


In [ ]:
import importlib
import sys


def reload_ramanv2():
    """清除 ramanv2 模块缓存，以加载当前目录中的最新代码。"""
    for name in list(sys.modules):
        if name == "ramanv2" or name.startswith("ramanv2."):
            del sys.modules[name]
    importlib.invalidate_caches()
    config_module = importlib.import_module("ramanv2.core.config")
    print("ramanv2 modules reloaded.")
    return config_module


config_module = reload_ramanv2()
build_config = config_module.build_config


In [ ]:
from ramanv2.packaging.platform_bundle import DATA_ARCHIVE_NAME, restore_data_archive

# 首次运行或更换 data.npz 后执行一次；会整体替换数据源的 init 与 init_test。
# 日常修改 notebook 设置不需要重跑本格。
DATA_ARCHIVE = PROJECT_ROOT / DATA_ARCHIVE_NAME
if DATA_ARCHIVE.is_file():
    restore_report = restore_data_archive(DATA_ARCHIVE, PROJECT_ROOT)
    print(
        f"restored {DATA_ARCHIVE.name}: samples={restore_report['sample_count']} "
        f"stages={restore_report['sample_counts']}"
    )
else:
    print(f"未找到 {DATA_ARCHIVE.name}，使用已有 dataset 目录")


## 2. 数据集与实验配置

In [ ]:
from pathlib import Path

import yaml
from ramanv2.core.config import build_config
from ramanv2.data.datasets import load_dataset_registry
from ramanv2.data.profiles import resolve_selection

# 本格只决定数据集与实验级配置；可反复调整，不会重新解压数据。

# ===== 1. 数据源与运行覆盖项 =====
# 可用数据集会在下方自动打印（PB=病原菌，FBB=食源菌）。
DATASET_ID = "PB"
# 填写整数时覆盖本次运行的随机种子；None 使用 config.py 的默认值。
SEED = None
# 填写整数时覆盖训练 batch_size；None 使用默认值（64）。
# 复用 EXPERIMENT_DIR 时以该实验的 shared_config.yaml 为准，本项会被忽略。
BATCH_SIZE = None
# 实验根存在时直接复用其中的 shared_config.yaml；留空则由训练格新建时间戳目录。
EXPERIMENT_DIR = None
LAST_TRAINED_SLOT_DIR = None
# 跨平台常用覆盖项：显存或 CPU 核数差异较大时按需打开。
CONFIG_OVERRIDES = {
    # "train_loader_num_workers": 8,
    # "val_loader_num_workers": 8,
}

# ===== 2. 解析数据集与当前数据源 =====
dataset_root = PROJECT_ROOT / "dataset"
registry = load_dataset_registry(dataset_root)
for item in registry.list_datasets():
    print(
        f"{item.dataset_id:6} {item.display_name}  dir={item.directory.name}  "
        f"levels={item.hierarchy.level_names}"
    )

selection = resolve_selection(DATASET_ID, None, PROJECT_ROOT)
spec = selection.spec
profile = selection.profile
dataset_dir = selection.dataset_dir
PROFILE_ID = profile.profile_id
INIT_DIR = dataset_dir / profile.root_init
PROFILE_TEST_INIT_DIR = dataset_dir / profile.root_init_test

# ===== 3. 构建当前运行配置 =====
config_values = {
    "dataset_id": spec.dataset_id,
    "profile_id": PROFILE_ID,
    "label_schema": selection.hierarchy.label_schema,
    "bad_bands": spec.bad_bands,
}
if SEED is not None:
    config_values["seed"] = int(SEED)
if BATCH_SIZE is not None:
    config_values["batch_size"] = int(BATCH_SIZE)
config_values.update(CONFIG_OVERRIDES)
config = build_config(config_values)
if EXPERIMENT_DIR is not None:
    experiment_snapshot = Path(EXPERIMENT_DIR) / "shared_config.yaml"
    if not experiment_snapshot.is_file():
        raise FileNotFoundError(f"实验目录缺少 shared_config.yaml：{experiment_snapshot}")
    config = build_config(yaml.safe_load(experiment_snapshot.read_text(encoding="utf-8")))
    print(f"复用实验配置：{experiment_snapshot}")
print(
    f"\nselected = {spec.dataset_id} / {spec.display_name}"
    f"  hierarchy={selection.hierarchy.level_names}  bad_bands={spec.bad_bands}"
    f"  batch_size={config.training.batch_size}"
)
print(f"dataset_dir = {dataset_dir}")
print("训练目标（LEVEL_NAME / PARENT_INDEX / GLOBAL_HEAD_ENABLE）请在“训练”格设置。")

## 3. 构建训练集与测试集

In [ ]:
from ramanv2.data.builders.train_test import build_test, build_train
from ramanv2.data.count import count_dataset, print_count_results

# BUILD_TRAIN_ENABLE：从 init 构建 train。
BUILD_TRAIN_ENABLE = True
# BUILD_TEST_ENABLE：从 init_test 构建 test；没有 init_test 时自动跳过。
BUILD_TEST_ENABLE = True
# SHOW_COUNTS_ENABLE：打印 train/test 的目录树和数量。
SHOW_COUNTS_ENABLE = True

if BUILD_TRAIN_ENABLE:
    build_train(profile, dataset_dir, input_config=config.input, hierarchy=selection.hierarchy)

if BUILD_TEST_ENABLE:
    if PROFILE_TEST_INIT_DIR.is_dir():
        build_test(
            profile,
            dataset_dir,
            input_config=config.input,
            test_root=PROFILE_TEST_INIT_DIR,
            hierarchy=selection.hierarchy,
        )
    else:
        print(f"未找到 {PROFILE_TEST_INIT_DIR}，跳过独立测试集构建。")

if SHOW_COUNTS_ENABLE:
    for label, stage_dir in (
        ("train", dataset_dir / profile.root_train_clean),
        ("test", dataset_dir / profile.root_test),
    ):
        if stage_dir.is_dir():
            tree, sample_count = count_dataset(stage_dir)
            print(f"\n[{label}]")
            print_count_results(tree, sample_count)
        else:
            print(f"\n[{label}] 目录不存在：{stage_dir}")

## 4. 实验组与训练

In [ ]:
from datetime import datetime
from pathlib import Path

import yaml

from ramanv2.core.config import build_config
from ramanv2.data.experiment_group import build_experiment_group

# 只在需要多个实验共享同一份 train/val 切分时运行；普通扁平训练可以跳过。
PREPARE_EXPERIMENT_GROUP_ENABLE = False  # 要走层级/共享切分时设为 True
# 自动时间戳：每次运行生成一个新实验组目录。
GROUP_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
# PB 使用 ("hier", "genus", "species")；扁平数据集例如 FBB 使用 ("main",)。
GROUP_EXPERIMENTS = ("hier", "genus", "species")
# GROUP_SEED 只影响 canonical train/val 切分，与模型训练 seed 分开。
GROUP_SEED = 42
GROUP_TRAIN_RATIO = 0.8
# 准备完成后自动切换到该实验并加载其 shared_config。
ACTIVE_EXPERIMENT = "hier"

if PREPARE_EXPERIMENT_GROUP_ENABLE:
    group_report = build_experiment_group(
        profile_id=profile.profile_id,
        tag=GROUP_TAG,
        experiment_names=GROUP_EXPERIMENTS,
        seed=GROUP_SEED,
        train_ratio=GROUP_TRAIN_RATIO,
        # 把当前 notebook 配置（含 BATCH_SIZE/seed/覆盖项）写入实验快照。
        config=config,
    )
    print(
        f"dataset = {group_report['profile_id']}  "
        f"samples = {group_report['sample_count']}  "
        f"train = {group_report['train_count']}  val = {group_report['val_count']}"
    )
    for name, item in group_report["experiments"].items():
        print(f"  {name:8} {item['experiment_dir']}")
        print(f"    train_sha256 = {item['train_split_sha256']}")
    active_item = group_report["experiments"].get(ACTIVE_EXPERIMENT)
    if active_item is None:
        raise KeyError(f"实验组中没有 {ACTIVE_EXPERIMENT}：{list(group_report['experiments'])}")
    # 自动把后续训练/评估指向选中的实验目录。
    EXPERIMENT_DIR = active_item["experiment_dir"]
    snapshot = Path(EXPERIMENT_DIR) / "shared_config.yaml"
    config = build_config(yaml.safe_load(snapshot.read_text(encoding="utf-8")))
    LAST_TRAINED_SLOT_DIR = None
    print(f"active_experiment = {ACTIVE_EXPERIMENT}")
    print(f"EXPERIMENT_DIR = {EXPERIMENT_DIR}")
    print("已加载该实验的 shared_config，可直接运行训练格。")
    print("切换 genus/species：将配置格 EXPERIMENT_DIR 改为对应目录并重跑配置格。")


In [ ]:
from ramanv2.data.runtime.index import DatasetIndex
from ramanv2.training.scope import collect_parent_preview, resolve_training_parent_ids

# 训练前只读预览：查看每层有哪些父类、哪些需要训练、单子类会直通。
train_dir = dataset_dir / profile.root_train_clean
if selection.hierarchy.is_flat or len(selection.hierarchy.level_names) < 2:
    print("当前数据集没有上级层级，PARENT_INDEX 保持 None。")
elif not train_dir.is_dir():
    print(f"训练目录不存在：{train_dir}，请先构建训练集。")
else:
    dataset_index = DatasetIndex(train_dir, selection.hierarchy, load_intensity_enable=False)
    for row in collect_parent_preview(dataset_index):
        note = "需训练" if row.trainable else "单子类直通"
        labels = ", ".join(row.child_labels)
        print(
            f"{row.level_name}  parent={row.parent_id:>2}  {row.parent_name:<20}  "
            f"子类 {row.child_count:>2}  {note}  {labels}"
        )
    # 每个层级留空 PARENT_INDEX 时会批量训练这些可训练父类。
    for level_name in dataset_index.level_names[1:]:
        parent_ids = resolve_training_parent_ids(dataset_index, level_name)
        class_count = len(dataset_index.get_class_names(level_name))
        print(f"{level_name} 可训练 PARENT_INDEX：{parent_ids}；全局类别数：{class_count}")


In [ ]:
from datetime import datetime

from ramanv2.common.paths import resolve_path
from ramanv2.data.runtime.index import DatasetIndex
from ramanv2.training.scope import resolve_training_parent_ids
from ramanv2.training.session import run_training_scope

# ===== 本次训练目标：只改这三项 =====
LEVEL_NAME = "level_1"
# 非首层留空表示批量训练该层全部可训练父类；填写整数只训练一个父类。
PARENT_INDEX = None
# True 表示训练该层全部类别的全局模型（平铺 head），仅用于 level_2 及以上。
GLOBAL_HEAD_ENABLE = False

# EXPERIMENT_DIR 为空时新建时间戳目录；否则复用已有实验的 shared_config 和 split。
if EXPERIMENT_DIR is None:
    active_experiment_dir = (
        PROJECT_ROOT
        / "output"
        / selection.spec.directory.name
        / dataset_dir.name
        / datetime.now().strftime("%Y%m%d_%H%M%S")
    )
else:
    active_experiment_dir = resolve_path(EXPERIMENT_DIR)
EXPERIMENT_DIR = active_experiment_dir

train_dir = dataset_dir / profile.root_train_clean
if not train_dir.is_dir():
    raise FileNotFoundError(f"训练目录不存在：{train_dir}，请先构建训练集")
dataset_index = DatasetIndex(train_dir, selection.hierarchy, load_intensity_enable=False)
parent_ids = resolve_training_parent_ids(
    dataset_index,
    LEVEL_NAME,
    PARENT_INDEX,
    GLOBAL_HEAD_ENABLE,
)
if not parent_ids:
    print(f"{LEVEL_NAME} 没有需要训练的父类（单子类父类均直通）。")
    LAST_TRAINED_SLOT_DIR = None
else:
    # run_training_scope 内部循环 parent_ids，并校验实验级 train_data.npz。
    scope_result = run_training_scope(
        config,
        active_experiment_dir,
        LEVEL_NAME,
        parent_ids,
        GLOBAL_HEAD_ENABLE,
    )
    LAST_TRAINED_SLOT_DIR = scope_result.last_slot_dir
    print("已训练槽位：")
    for slot_dir in scope_result.slot_dirs:
        print(f"  {slot_dir}")
    print(f"last_trained_slot_dir = {LAST_TRAINED_SLOT_DIR}")
    if scope_result.snapshot_path is not None:
        print(
            f"train snapshot: {scope_result.snapshot_path}; "
            f"samples={scope_result.sample_count}; shape={scope_result.intensity_shape}"
        )


## 5. 训练后评估

In [ ]:
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.slot_flow import evaluate_slot_with_baseline

# 默认评估刚训练完的槽位；也可以手动指定其他槽位目录。
SINGLE_SLOT_DIR = LAST_TRAINED_SLOT_DIR
SINGLE_EVALUATE_ENABLE = True
# GPU 显存不足或需要稳定复现时设为 True。
CPU_ENABLE = False

if SINGLE_EVALUATE_ENABLE:
    if SINGLE_SLOT_DIR is None:
        raise ValueError("请先完成训练，或手动填写 SINGLE_SLOT_DIR")
    # 自动从实验元数据识别 level/parent，并同时跑模型和 PCA-SVM 基线。
    evaluation = evaluate_slot_with_baseline(
        Path(SINGLE_SLOT_DIR),
        "cpu" if CPU_ENABLE else None,
    )
    print(f"slot_dir   = {evaluation.slot_dir}")
    print(f"level_name = {evaluation.level_name}  parent_id = {evaluation.parent_id}")
    for name, result_dir, metrics in (
        ("模型", evaluation.model_result_dir, evaluation.model_metrics),
        ("PCA-SVM 基线", evaluation.baseline_result_dir, evaluation.baseline_metrics),
    ):
        summary = metrics["summary"]
        print(f"\n{name} 验证集指标")
        print(f" Val Set Accuracy:  {summary['accuracy'] * 100:.4f}%")
        print(f" Macro F1-score:    {summary['macro_f1'] * 100:.4f}%")
        print(f" Macro Recall:      {summary['macro_recall'] * 100:.4f}%")
        figure_path = result_dir / "confusion_matrix.png"
        print(f"{name} 混淆矩阵：{figure_path}")
        display(Image(filename=str(figure_path)))


In [ ]:
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.independent_test import evaluate_test_set
from ramanv2.evaluation.slot_flow import format_test_report

# 默认评估刚训练完的槽位；测试集默认为 dataset/<数据集>/test。
TEST_EVAL_ENABLE = True
TEST_SLOT_DIR = LAST_TRAINED_SLOT_DIR
CPU_ENABLE = False
# 覆盖测试集目录；None 使用当前数据源的 test。
TEST_DIR_OVERRIDE = None
# 每个测试文件夹抽样多少条；None 表示全部样本。
TEST_SAMPLE_LIMIT = None

if TEST_EVAL_ENABLE:
    if TEST_SLOT_DIR is None:
        raise ValueError("请先完成训练，或手动填写 TEST_SLOT_DIR")
    test_dir = (
        Path(TEST_DIR_OVERRIDE)
        if TEST_DIR_OVERRIDE
        else dataset_dir / profile.root_test
    )
    if not test_dir.is_dir():
        print(f"未找到预处理后的测试集：{test_dir}，请先执行构建测试集。")
    else:
        test_report = evaluate_test_set(
            TEST_SLOT_DIR,
            "cpu" if CPU_ENABLE else None,
            test_dir,
            TEST_SAMPLE_LIMIT,
        )
        # format_test_report 复用 evaluation.summary_text 的统一格式。
        print()
        print(format_test_report(test_report))
        result_dir = Path(test_report["result_dir"])
        print(f"metrics.json = {result_dir / 'metrics.json'}")
        display(Image(filename=str(result_dir / "confusion_matrix.png")))


## 6. 层级汇总

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.level_summary import level_summary_availability, summarize_level_results

# 层级汇总只适用于有父类槽位的层级；level_1 和全局 head 实验会跳过。
SUMMARY_LEVEL = "level_2"
SUMMARY_ENABLE = True
SUMMARY_CPU_ENABLE = False
# 缺失的父类 val/test 结果会自动评估一次。
SUMMARY_AUTO_EVALUATE = True
MODE_TITLES = {"parent_known": "已知上层", "cascade": "从头级联"}

summary_available, summary_reason = (
    level_summary_availability(EXPERIMENT_DIR, SUMMARY_LEVEL)
    if EXPERIMENT_DIR is not None
    else (False, "请先完成训练或填写 EXPERIMENT_DIR")
)
if SUMMARY_ENABLE and not summary_available:
    print(f"跳过层级汇总：{summary_reason}")
elif SUMMARY_ENABLE:
    summary_report = summarize_level_results(
        EXPERIMENT_DIR,
        SUMMARY_LEVEL,
        "cpu" if SUMMARY_CPU_ENABLE else None,
        auto_evaluate_enable=SUMMARY_AUTO_EVALUATE,
    )
    for mode in ("parent_known", "cascade"):
        for output_name, title in (("val_result", "验证"), ("test_result", "测试")):
            result_dir = Path(summary_report["level_summary_dir"]) / mode / output_name
            metrics_path = result_dir / "metrics.json"
            if not metrics_path.is_file():
                print(f"{MODE_TITLES[mode]} {title}结果尚未生成：{result_dir}")
                continue
            summary = json.loads(metrics_path.read_text(encoding="utf-8"))["summary"]
            print(f"\n===== {MODE_TITLES[mode]} · {title} =====")
            print(f"Accuracy = {summary['accuracy'] * 100:.4f}%  (support={summary['support']})")
            print(f"Macro F1 = {summary['macro_f1'] * 100:.4f}%")
            print(f"Macro Recall = {summary['macro_recall'] * 100:.4f}%")
            print(f"result_dir = {result_dir}")
            display(Image(filename=str(result_dir / "confusion_matrix.png")))


## 7. 单模型 Analysis

In [ ]:
from IPython.display import Image, display
from ramanv2.analysis.interpret import interpret_slot
from ramanv2.evaluation.context import resolve_slot_target

# 默认分析最近训练的槽位；优先使用单模型评估格里的 SINGLE_SLOT_DIR。
SINGLE_ANALYZE_ENABLE = True
# 显存不足时设为 True；UMAP/IG 在 CPU 上较慢但不会 OOM。
CPU_ENABLE = False
# 归因只抽取最多 5 个 batch；IG 步数沿用实验配置。
ANALYSIS_ATTRIBUTION_BATCH_COUNT = 5

resolved_single_slot_dir = globals().get("SINGLE_SLOT_DIR") or globals().get("LAST_TRAINED_SLOT_DIR")
if SINGLE_ANALYZE_ENABLE:
    if resolved_single_slot_dir is None:
        raise ValueError("请先训练一个唯一模型，或填写 SINGLE_SLOT_DIR")
    single_level, single_parent = resolve_slot_target(resolved_single_slot_dir)
    print(f"slot_dir   = {resolved_single_slot_dir}")
    print(f"level_name = {single_level}  parent_id = {single_parent}")
    analysis_dir = interpret_slot(
        resolved_single_slot_dir,
        single_level,
        "cpu" if CPU_ENABLE else None,
        attribution_batch_count=ANALYSIS_ATTRIBUTION_BATCH_COUNT,
    )
    print(analysis_dir)
    for figure_name in ("channel_importance_IG.png", "layer_importance.png"):
        display(Image(filename=str(analysis_dir / "figures" / figure_name)))
    log_path = analysis_dir / "logs" / "analysis_log.txt"
    if log_path.is_file():
        print("\n===== Analysis 摘要 =====")
        print(log_path.read_text(encoding="utf-8"))


## 8. 打包实验结果

In [ ]:
from pathlib import Path

from ramanv2.packaging.experiment_archive import publish_experiment_archive

# False 时只打印实验目录路径，不生成压缩包。
PACKAGE_RESULT_ENABLE = True

if EXPERIMENT_DIR is None:
    raise ValueError("请先完成训练或填写 EXPERIMENT_DIR")
result_archive = publish_experiment_archive(
    Path(EXPERIMENT_DIR).resolve(),
    package_enable=PACKAGE_RESULT_ENABLE,
)
